# Transformer Encoder untuk Intent Classification (Bahasa Indonesia)

Lanjutan dari notebook sebelumnya. Versi awal berhasil jalan end-to-end tapi menunjukkan tanda **overfitting** yang jelas: training loss turun ke ~0 sejak epoch ke-9 dari 20, sementara test set cuma 19 sampel (jadi 1 sampel = ~5,3 poin akurasi, terlalu noisy untuk jadi acuan).

Notebook ini menambahkan investigasi dan perbaikan untuk masalah itu:
- **Baseline** — reproduksi kondisi overfitting dari notebook sebelumnya, buat pembanding
- **3 varian model dibandingkan pakai 5-fold Stratified Cross-Validation** (bukan cuma 1 split), supaya angka akurasi lebih stabil:
  1. `scratch_large` — Transformer dari nol, konfigurasi awal (6 layer, hidden 768) → baseline yang overfit
  2. `scratch_small` — Transformer dari nol, kapasitas diperkecil (2 layer, hidden 128, dropout lebih tinggi) + weight decay
  3. `finetuned_bert` — fine-tune BERT Indonesia yang sudah di-pretrain (`AutoModelForSequenceClassification`), bukan cuma pakai tokenizer-nya seperti sebelumnya
- Semua varian pakai **validation split + early stopping** berdasarkan validation loss, bukan epoch tetap
- Model dengan rata-rata CV accuracy terbaik dilatih ulang di seluruh data buat jadi model final yang di-serve

*Catatan: semua angka di notebook ini masih placeholder sampai benar-benar dijalankan di Colab. Disarankan pakai runtime GPU karena ada bagian fine-tuning BERT.*

## Struktur notebook
1. Setup & Install
2. Load & Preprocessing Dataset
3. Tokenizer
4. Dataset & DataLoader
5. Arsitektur Model
6. Baseline (reproduksi overfitting)
7. Perbandingan 3 varian dengan 5-fold Cross-Validation
8. Latih model final (varian terbaik) di seluruh data
9. Simpan Model
10. Serving (FastAPI + ngrok)
11. Uji Coba Endpoint

## 1. Setup & Install

In [1]:
!pip install -q transformers fastapi uvicorn pyngrok pydantic torch scikit-learn pandas numpy

In [2]:
import string
import json
import os
import requests
import threading
import math
import time
from getpass import getpass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.functional import softmax
from torch.utils.data import DataLoader, Dataset

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 2. Load & Preprocessing Dataset

In [3]:
DATA_URL = "https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/Hands-On-NLP-Super-Class-Batch1/refs/heads/main/Transformer/data3.csv"
data = pd.read_csv(DATA_URL)

def preprocess_text(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation))

data['text'] = data['text'].apply(preprocess_text)
data.head()

,text,intent
0,siapa nama kamu,siapa_anda
1,kamu siapa,siapa_anda
2,ceritakan tentang dirimu,siapa_anda
3,apa kamu bisa memperkenalkan diri,siapa_anda
4,siapa yang membuatmu,siapa_anda


In [4]:
X = data['text'].tolist()
y_categorical = data['intent'].astype('category')
label_map = {label: idx for idx, label in enumerate(y_categorical.cat.categories)}
y = y_categorical.map(label_map).tolist()

print("Label map:", label_map)
print("Jumlah data per kelas:")
print(data['intent'].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Label map: {'greeting': 0, 'sekarang_jam_berapa': 1, 'siapa_anda': 2}
Jumlah data per kelas:
intent
greeting               33
siapa_anda             32
sekarang_jam_berapa    29
Name: count, dtype: int64


## 3. Tokenizer

Vocab size diambil dinamis dari `tokenizer.vocab_size` (32000 untuk model ini), bukan di-hardcode.

In [5]:
MODEL_NAME = "cahya/bert-base-indonesian-522M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
VOCAB_SIZE = tokenizer.vocab_size
MAX_LENGTH = 128
NUM_LABELS = len(label_map)

print("Vocab size:", VOCAB_SIZE)
print("Jumlah label:", NUM_LABELS)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/230k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Vocab size: 32000
Jumlah label: 3


## 4. Dataset & DataLoader

In [6]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        tokens = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }

## 5. Arsitektur Model

- **PositionalEncoding** — rumus sinusoidal sesuai materi.
- **TransformerClassifier** — encoder-only dari nol, sekarang parameternya (jumlah layer, hidden dim, dropout) bisa diatur, supaya bisa dibikin versi besar (baseline) dan versi kecil (regularized) dari class yang sama.
- Untuk pendekatan transfer learning, dipakai `AutoModelForSequenceClassification` langsung dari HuggingFace (fine-tune bobot BERT yang sudah pretrained, bukan cuma tokenizer-nya).

In [7]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding:
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_length=512):
        super().__init__()
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :].unsqueeze(0)

In [8]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, hidden_dim=768, nhead=8,
                 num_layers=6, max_length=MAX_LENGTH, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_encoding = PositionalEncoding(hidden_dim, max_length)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        embeddings = self.embedding(input_ids)
        embeddings = self.positional_encoding(embeddings)
        embeddings = self.dropout(embeddings)
        transformer_out = self.transformer(
            embeddings.permute(1, 0, 2),
            src_key_padding_mask=(attention_mask == 0),
        )
        pooled_output = transformer_out.mean(dim=0)
        logits = self.classifier(pooled_output)
        return logits

## 6. Baseline (reproduksi overfitting)

Ini konfigurasi dan cara training yang sama seperti notebook sebelumnya (tanpa validation split, tanpa early stopping) — sengaja dipertahankan sebagai pembanding "sebelum diperbaiki".

In [9]:
baseline_train_dataset = IntentDataset(X_train, y_train, tokenizer)
baseline_test_dataset = IntentDataset(X_test, y_test, tokenizer)
baseline_train_loader = DataLoader(baseline_train_dataset, batch_size=32, shuffle=True)
baseline_test_loader = DataLoader(baseline_test_dataset, batch_size=32, shuffle=False)

baseline_model = TransformerClassifier(vocab_size=VOCAB_SIZE, num_labels=NUM_LABELS).to(device)
baseline_optimizer = optim.AdamW(baseline_model.parameters(), lr=5e-5)
baseline_criterion = nn.CrossEntropyLoss()

BASELINE_EPOCHS = 20
baseline_history = []
for epoch in range(BASELINE_EPOCHS):
    baseline_model.train()
    total_loss = 0
    for batch in baseline_train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        baseline_optimizer.zero_grad()
        logits = baseline_model(input_ids, attention_mask)
        loss = baseline_criterion(logits, labels)
        loss.backward()
        baseline_optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(baseline_train_loader)
    baseline_history.append(avg_loss)
    print(f"Epoch {epoch + 1}/{BASELINE_EPOCHS}, Loss: {avg_loss:.4f}")

/tmp/ipykernel_4285/3507138452.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch 1/20, Loss: 1.3254
Epoch 2/20, Loss: 0.6036
Epoch 3/20, Loss: 0.3295
Epoch 4/20, Loss: 0.1540
Epoch 5/20, Loss: 0.0859
Epoch 6/20, Loss: 0.0164
Epoch 7/20, Loss: 0.0259
Epoch 8/20, Loss: 0.0256
Epoch 9/20, Loss: 0.0028
Epoch 10/20, Loss: 0.0008
Epoch 11/20, Loss: 0.0007
Epoch 12/20, Loss: 0.0009
Epoch 13/20, Loss: 0.0008
Epoch 14/20, Loss: 0.0009
Epoch 15/20, Loss: 0.0019
Epoch 16/20, Loss: 0.0009
Epoch 17/20, Loss: 0.0006
Epoch 18/20, Loss: 0.0007
Epoch 19/20, Loss: 0.0004
Epoch 20/20, Loss: 0.0004


In [10]:
baseline_model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in baseline_test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = baseline_model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds)

print("Baseline test accuracy (1 split, 19 sampel, TANPA early stopping):", accuracy_score(y_true, y_pred))
print("\nCatatan: bandingkan loss curve di atas dengan hasil cross-validation di bagian berikutnya.")

Baseline test accuracy (1 split, 19 sampel, TANPA early stopping): 0.9473684210526315

Catatan: bandingkan loss curve di atas dengan hasil cross-validation di bagian berikutnya.


## 7. Perbandingan 3 Varian dengan 5-Fold Cross-Validation

Fungsi generik di bawah menangani training untuk dua jenis model sekaligus:
- `model_type="scratch"` — `TransformerClassifier` custom, loss dihitung manual pakai `CrossEntropyLoss`
- `model_type="finetuned_bert"` — `AutoModelForSequenceClassification`, loss dihitung otomatis oleh model HuggingFace lewat argumen `labels`

Semua varian pakai early stopping: training berhenti kalau validation loss nggak membaik selama `patience` epoch berturut-turut, lalu bobot terbaik (val loss terendah) yang dipakai.

In [11]:
def forward_and_loss(model, batch, criterion, model_type):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    if model_type == "finetuned_bert":
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
    else:
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

    return loss, logits, labels


def train_with_early_stopping(model, model_type, train_loader, val_loader,
                               max_epochs=30, patience=5, lr=None, weight_decay=0.01):
    criterion = nn.CrossEntropyLoss()
    default_lr = 2e-5 if model_type == "finetuned_bert" else 5e-5
    optimizer = optim.AdamW(model.parameters(), lr=lr or default_lr, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(max_epochs):
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            loss, _, _ = forward_and_loss(model, batch, criterion, model_type)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                loss, _, _ = forward_and_loss(model, batch, criterion, model_type)
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def evaluate_model(model, loader, model_type):
    model.eval()
    y_true, y_pred = [], []
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            _, logits, labels = forward_and_loss(model, batch, criterion, model_type)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds)
    return accuracy_score(y_true, y_pred), y_true, y_pred

In [12]:
SCRATCH_VARIANTS = {
    "scratch_large": dict(hidden_dim=768, nhead=8, num_layers=6, dropout=0.1),
    "scratch_small": dict(hidden_dim=128, nhead=4, num_layers=2, dropout=0.3),
}

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

X_arr = np.array(X, dtype=object)
y_arr = np.array(y)

cv_results = {name: [] for name in list(SCRATCH_VARIANTS.keys()) + ["finetuned_bert"]}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_arr, y_arr)):
    print(f"\n=== Fold {fold + 1}/{N_SPLITS} ===")
    X_tr, X_val = X_arr[train_idx].tolist(), X_arr[val_idx].tolist()
    y_tr, y_val = y_arr[train_idx].tolist(), y_arr[val_idx].tolist()

    train_ds = IntentDataset(X_tr, y_tr, tokenizer)
    val_ds = IntentDataset(X_val, y_val, tokenizer)
    train_loader_cv = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader_cv = DataLoader(val_ds, batch_size=8, shuffle=False)

    for name, cfg in SCRATCH_VARIANTS.items():
        torch.manual_seed(42)
        model_cv = TransformerClassifier(vocab_size=VOCAB_SIZE, num_labels=NUM_LABELS, **cfg).to(device)
        model_cv, _ = train_with_early_stopping(
            model_cv, "scratch", train_loader_cv, val_loader_cv,
            max_epochs=30, patience=5,
        )
        acc, _, _ = evaluate_model(model_cv, val_loader_cv, "scratch")
        cv_results[name].append(acc)
        print(f"  {name}: val accuracy = {acc:.4f}")

    torch.manual_seed(42)
    bert_cv = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
    bert_cv, _ = train_with_early_stopping(
        bert_cv, "finetuned_bert", train_loader_cv, val_loader_cv,
        max_epochs=10, patience=3, lr=2e-5,
    )
    acc, _, _ = evaluate_model(bert_cv, val_loader_cv, "finetuned_bert")
    cv_results["finetuned_bert"].append(acc)
    print(f"  finetuned_bert: val accuracy = {acc:.4f}")


=== Fold 1/5 ===


/tmp/ipykernel_4285/3507138452.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  scratch_large: val accuracy = 1.0000


/tmp/ipykernel_4285/3507138452.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  scratch_small: val accuracy = 1.0000


...

/tmp/ipykernel_4285/3507138452.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  scratch_small: val accuracy = 1.0000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cahya/bert-base-indonesian-522M
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  finetuned_bert: val accuracy = 1.0000


In [13]:
print("=== Ringkasan 5-Fold Cross-Validation ===\n")
for name, accs in cv_results.items():
    print(f"{name}: {np.mean(accs):.4f} ± {np.std(accs):.4f}  | per fold: {[round(a, 3) for a in accs]}")

best_variant = max(cv_results, key=lambda k: np.mean(cv_results[k]))
print(f"\nVarian dengan rata-rata CV accuracy tertinggi: {best_variant}")
print("(Catatan: angka di atas baru valid setelah cell ini benar-benar dijalankan)")

=== Ringkasan 5-Fold Cross-Validation ===

scratch_large: 0.9895 ± 0.0211  | per fold: [1.0, 1.0, 1.0, 0.947, 1.0]
scratch_small: 0.9789 ± 0.0258  | per fold: [1.0, 0.947, 1.0, 0.947, 1.0]
finetuned_bert: 1.0000 ± 0.0000  | per fold: [1.0, 1.0, 1.0, 1.0, 1.0]

Varian dengan rata-rata CV accuracy tertinggi: finetuned_bert
(Catatan: angka di atas baru valid setelah cell ini benar-benar dijalankan)


## 8. Latih Model Final (Varian Terbaik) di Seluruh Data

Varian dengan rata-rata CV accuracy tertinggi dilatih ulang, kali ini dengan train/validation split dari seluruh dataset (bukan cuma satu fold), supaya model final memanfaatkan data sebanyak mungkin. Validation split tetap dipertahankan untuk early stopping.

In [14]:
X_final_train, X_final_val, y_final_train, y_final_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

final_train_ds = IntentDataset(X_final_train, y_final_train, tokenizer)
final_val_ds = IntentDataset(X_final_val, y_final_val, tokenizer)
final_train_loader = DataLoader(final_train_ds, batch_size=8, shuffle=True)
final_val_loader = DataLoader(final_val_ds, batch_size=8, shuffle=False)

if best_variant == "finetuned_bert":
    model_type = "finetuned_bert"
    final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
    final_model, final_history = train_with_early_stopping(
        final_model, model_type, final_train_loader, final_val_loader,
        max_epochs=15, patience=4, lr=2e-5,
    )
    model_config = {"model_name": MODEL_NAME}
else:
    model_type = "scratch"
    cfg = SCRATCH_VARIANTS[best_variant]
    final_model = TransformerClassifier(
        vocab_size=VOCAB_SIZE, num_labels=NUM_LABELS, max_length=MAX_LENGTH, **cfg
    ).to(device)
    final_model, final_history = train_with_early_stopping(
        final_model, model_type, final_train_loader, final_val_loader,
        max_epochs=50, patience=7,
    )
    model_config = dict(vocab_size=VOCAB_SIZE, num_labels=NUM_LABELS, max_length=MAX_LENGTH, **cfg)

final_acc, y_true_final, y_pred_final = evaluate_model(final_model, final_val_loader, model_type)
print(f"Model final ({best_variant}) — validation accuracy: {final_acc:.4f}\n")
print(classification_report(y_true_final, y_pred_final, target_names=list(label_map.keys())))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cahya/bert-base-indonesian-522M
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model final (finetuned_bert) — validation accuracy: 1.0000

                     precision    recall  f1-score   support

           greeting       1.00      1.00      1.00         5
sekarang_jam_berapa       1.00      1.00      1.00         5
         siapa_anda       1.00      1.00      1.00         5

           accuracy                           1.00        15
          macro avg       1.00      1.00      1.00        15
       weighted avg       1.00      1.00      1.00        15



## 9. Simpan Model

Model, konfigurasi arsitektur (`model_type` + `model_config`), label map, dan hasil cross-validation semuanya disimpan bareng dalam satu folder, supaya bagian serving tahu persis jenis model apa yang harus di-load ulang.

In [19]:
SAVE_DIR = "intent_transformer_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

if model_type == "finetuned_bert":
    final_model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
else:
    torch.save(final_model.state_dict(), os.path.join(SAVE_DIR, "model.pth"))

# Save notebook-specific metadata to metadata.json
with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump({
        "label_map": label_map,
        "model_type": model_type,
        "model_config": model_config,
        "best_variant": best_variant,
        "cv_results": {
            name: {
                "mean": float(np.mean(accs)),
                "std": float(np.std(accs)),
                "per_fold": [float(a) for a in accs],
            }
            for name, accs in cv_results.items()
        },
    }, f, indent=2)

print("Tersimpan di:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tersimpan di: intent_transformer_artifacts


> Opsional: kalau mau artifact tetap ada setelah runtime Colab berakhir, salin folder `intent_transformer_artifacts/` ke Google Drive. Sel di bawah ini opsional dan tidak dijalankan otomatis.

In [16]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(SAVE_DIR, "/content/drive/My Drive/intent_transformer_artifacts", dirs_exist_ok=True)

## 10. Serving (FastAPI + ngrok)

Loading model disesuaikan otomatis berdasarkan `model_type` yang disimpan di `config.json` — jadi kalau varian terbaiknya ternyata `finetuned_bert`, kode ini otomatis load lewat `AutoModelForSequenceClassification`; kalau varian scratch, otomatis rekonstruksi `TransformerClassifier` dengan konfigurasi yang tersimpan.

In [20]:
with open(os.path.join(SAVE_DIR, "metadata.json")) as f:
    artifacts = json.load(f)

serving_label_map = {int(v): k for k, v in artifacts["label_map"].items()}
serving_model_type = artifacts["model_type"]

if serving_model_type == "finetuned_bert":
    serving_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
    serving_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
else:
    serving_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    serving_model = TransformerClassifier(**artifacts["model_config"]).to(device)
    serving_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "model.pth"), map_location=device))

serving_model.eval()
print("Model type untuk serving:", serving_model_type)
print("Label map:", serving_label_map)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model type untuk serving: finetuned_bert
Label map: {0: 'greeting', 1: 'sekarang_jam_berapa', 2: 'siapa_anda'}


In [21]:
app = FastAPI(title="Intent Classification API", version="1.0")

class TextRequest(BaseModel):
    text: str

class BatchRequest(BaseModel):
    texts: List[str]

class PredictionResponse(BaseModel):
    intent: str
    confidence: float

def predict_single(text: str) -> PredictionResponse:
    tokens = serving_tokenizer(
        text, padding="max_length", truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    with torch.no_grad():
        if serving_model_type == "finetuned_bert":
            outputs = serving_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
        else:
            logits = serving_model(input_ids, attention_mask)
        probabilities = softmax(logits, dim=1)
        confidence, predicted_label = torch.max(probabilities, dim=1)

    intent = serving_label_map[predicted_label.item()]
    return PredictionResponse(intent=intent, confidence=confidence.item())

@app.post("/predict", response_model=PredictionResponse)
async def predict(request: TextRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    return predict_single(request.text)

@app.post("/predict_batch", response_model=List[PredictionResponse])
async def predict_batch(request: BatchRequest):
    if not request.texts:
        raise HTTPException(status_code=400, detail="Text list cannot be empty.")
    return [predict_single(t) for t in request.texts]

@app.get("/")
async def health_check():
    return {"message": "Intent Classification API is up and running!", "model_type": serving_model_type}

## 11. Jalankan Server & Uji Coba

Authtoken ngrok diminta lewat input saat runtime (`getpass`), tidak ditulis langsung di kode.

In [22]:
NGROK_AUTHTOKEN = getpass("Masukkan ngrok authtoken (tidak ditampilkan): ")
from pyngrok import ngrok
import uvicorn

ngrok.set_auth_token(NGROK_AUTHTOKEN)

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

Masukkan ngrok authtoken (tidak ditampilkan): ··········
Public URL: https://tactile-diploma-dejected.ngrok-free.dev


In [23]:
time.sleep(2)  # kasih waktu server nyala

resp = requests.post(f"{ngrok_tunnel.public_url}/predict", json={"text": "Siapa nama kamu?"})
print(resp.json())

INFO:     34.16.239.186:0 - "POST /predict HTTP/1.1" 200 OK
{'intent': 'siapa_anda', 'confidence': 0.9978358149528503}


In [24]:
resp = requests.post(
    f"{ngrok_tunnel.public_url}/predict_batch",
    json={"texts": ["Nama kamu siapa?", "Selamat malam kawan"]},
)
print(resp.json())

INFO:     34.16.239.186:0 - "POST /predict_batch HTTP/1.1" 200 OK
[{'intent': 'siapa_anda', 'confidence': 0.9978476762771606}, {'intent': 'greeting', 'confidence': 0.9976987242698669}]


## Ringkasan Investigasi Overfitting

Metodologi yang dipakai di notebook ini untuk menjawab masalah overfitting dari versi sebelumnya:

- **Baseline dipertahankan** (bagian 6) sebagai bukti pembanding — training loss ~0 sejak epoch ke-9, test accuracy dari 1 split kecil (19 sampel) yang rentan noise.
- **3 varian dibandingkan dengan 5-fold Stratified Cross-Validation** dari bagian 7, supaya estimasi performa tidak bergantung ke satu split yang kebetulan mudah/susah:
  - `scratch_large` — konfigurasi awal, kemungkinan besar tetap overfit karena kapasitas model jauh lebih besar dari data
  - `scratch_small` — kapasitas diperkecil + dropout & weight decay lebih tinggi, buat menguji apakah regularisasi cukup
  - `finetuned_bert` — transfer learning beneran (bukan cuma tokenizer), buat menguji hipotesis bahwa representasi bahasa yang sudah dipelajari sebelumnya butuh lebih sedikit data untuk generalisasi
- Semua varian pakai **validation split + early stopping**, bukan jumlah epoch tetap, supaya training berhenti di titik performa validasi terbaik.
- Varian dengan rata-rata CV accuracy tertinggi dilatih ulang di seluruh data (bagian 8) untuk jadi model yang di-serve.

Hasil rata-rata ± standar deviasi tiap varian, dan varian mana yang akhirnya menang, baru bisa dilihat setelah notebook ini dijalankan penuh di Colab — angka-angka itu yang nanti masuk ke README dan blog post, bukan dugaan di sini.

---

*Notebook ini dibuat sebagai bagian dari pembelajaran mandiri, merujuk pada materi modul NLP rubythalib.ai (mentor: Muhammad Ikhwan Fathulloh). Kode, investigasi overfitting, dan perbaikan pada notebook ini merupakan hasil pengembangan mandiri dari materi asli.*